In [8]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


csv_file = 'P3-Car Market Trends Analysis with Car Dekho Data.csv'

if not os.path.exists(csv_file):
    raise FileNotFoundError(f"Dataset '{csv_file}' not found. Please verify the file path.")

df = pd.read_csv(csv_file)


df.drop_duplicates(inplace=True)
df.dropna(subset=['Present_Price', 'Selling_Price', 'Year'], inplace=True)


df['Depreciation'] = df['Present_Price'] - df['Selling_Price']
df['Depreciation_Pct'] = (df['Depreciation'] / df['Present_Price']) * 100
df['Car_Age'] = df['Year'].max() - df['Year']  # Age relative to newest car in dataset

# Summary Statistics Printout
print("=" * 50)
print("             CAR DEKHO EDA SUMMARY               ")
print("=" * 50)
print(f"Total Unique Records  : {len(df)}")
print(f"Average Selling Price : ₹{df['Selling_Price'].mean():.2f} Lakhs")
print(f"Average Present Price : ₹{df['Present_Price'].mean():.2f} Lakhs")
print(f"Average Depreciation  : {df['Depreciation_Pct'].mean():.1f}%")
print(f"Year Range            : {df['Year'].min()} - {df['Year'].max()}")
print(f"Fuel Types Available  : {', '.join(df['Fuel_Type'].unique())}")
print(f"Most Popular Model    : {df['Car_Name'].value_counts().index[0]}")
print("=" * 50)

# Detailed Numeric Inspection
print("\nNumerical Feature Breakdown:")
print(df[['Selling_Price', 'Present_Price', 'Kms_Driven', 'Depreciation_Pct', 'Car_Age']].describe().T[['mean', 'std', 'min', '50%', 'max']])
print("=" * 50)

# ==========================================
# 2. VISUALIZATION PIPELINE (DARK THEME)
# ==========================================
plt.rcParams['figure.facecolor'] = '#1a1a1a'
plt.rcParams['axes.facecolor'] = '#1a1a1a'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'

# Chart 1: Fuel Type Distribution
fuel_counts = df['Fuel_Type'].value_counts()
plt.figure(figsize=(8, 6))
plt.pie(fuel_counts.values, labels=fuel_counts.index, autopct='%1.1f%%',
        colors=['#4FC3F7', '#81C784', '#FFB74D'], startangle=90,
        textprops={'color': 'white'})
plt.title('Fuel Type Distribution', fontsize=16, fontweight='bold', color='white')
plt.savefig('chart1_fuel_type.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.close()

# Chart 2: Selling Price vs Present Price
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df['Present_Price'], df['Selling_Price'], c=df['Year'], cmap='cool', alpha=0.7, s=60)
plt.plot([0, df['Present_Price'].max()], [0, df['Present_Price'].max()], 'r--', alpha=0.5)
plt.xlabel('Present Price (₹ Lakhs)')
plt.ylabel('Selling Price (₹ Lakhs)')
plt.title('Selling Price vs Present Price', fontsize=16, fontweight='bold', color='white')
cbar = plt.colorbar(scatter)
cbar.set_label('Year', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')
plt.savefig('chart2_price_scatter.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.close()

# Chart 3: Year-wise Average Selling Price Trend
year_trend = df.groupby('Year')['Selling_Price'].mean().reset_index()
plt.figure(figsize=(10, 6))
plt.plot(year_trend['Year'], year_trend['Selling_Price'], marker='o', color='#4FC3F7', linewidth=2.5)
plt.fill_between(year_trend['Year'], year_trend['Selling_Price'], alpha=0.2, color='#4FC3F7')
plt.xlabel('Year')
plt.ylabel('Average Selling Price (₹ Lakhs)')
plt.title('Year-wise Average Selling Price Trend', fontsize=16, fontweight='bold', color='white')
plt.savefig('chart3_year_trend.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.close()

# Chart 4: Top 10 Car Models by Average Selling Price
top_models = df.groupby('Car_Name')['Selling_Price'].agg(['mean', 'count']).reset_index()
top_models = top_models[top_models['count'] >= 3].sort_values('mean', ascending=False).head(10)
plt.figure(figsize=(12, 6))
plt.barh(top_models['Car_Name'][::-1], top_models['mean'][::-1], color='#4FC3F7')
plt.xlabel('Average Selling Price (₹ Lakhs)')
plt.title('Top 10 Car Models by Average Selling Price', fontsize=16, fontweight='bold', color='white')
plt.savefig('chart4_top_models.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.close()

# Chart 5: Average Depreciation by Fuel Type
dep_fuel = df.groupby('Fuel_Type')['Depreciation_Pct'].mean().sort_values(ascending=False)
plt.figure(figsize=(10, 6))
plt.bar(dep_fuel.index, dep_fuel.values, color=['#E57373', '#4FC3F7', '#81C784'])
plt.ylabel('Average Depreciation (%)')
plt.title('Average Depreciation by Fuel Type', fontsize=16, fontweight='bold', color='white')
plt.savefig('chart5_depreciation.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.close()

# Chart 6: Correlation Heatmap
plt.figure(figsize=(10, 7))
corr_cols = [c for c in ['Year', 'Selling_Price', 'Present_Price', 'Kms_Driven', 'Depreciation_Pct'] if c in df.columns]
corr = df[corr_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f', cbar=True)
plt.title('Correlation Heatmap', fontsize=16, fontweight='bold', color='white')
plt.savefig('chart6_correlation.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.close()

print("EDA completed and all visualization PNG files saved successfully.")

             CAR DEKHO EDA SUMMARY               
Total Unique Records  : 299
Average Selling Price : ₹4.59 Lakhs
Average Present Price : ₹7.54 Lakhs
Average Depreciation  : 36.6%
Year Range            : 2003 - 2018
Fuel Types Available  : Petrol, Diesel, CNG
Most Popular Model    : city

Numerical Feature Breakdown:
                          mean           std         min           50%  \
Selling_Price         4.589632      4.984240    0.100000      3.510000   
Present_Price         7.541037      8.567887    0.320000      6.100000   
Kms_Driven        36916.752508  39015.170352  500.000000  32000.000000   
Depreciation_Pct     36.621974     20.287451    1.074547     34.574468   
Car_Age               4.384615      2.896868    0.000000      4.000000   

                            max  
Selling_Price         35.000000  
Present_Price         92.600000  
Kms_Driven        500000.000000  
Depreciation_Pct      89.464812  
Car_Age               15.000000  
EDA completed and all visualizat